# 02 - Data Cleaning

### Notebook Overview

Clean the raw dataset by dropping unnecessary columns, fixing data types, correcting typos, and extracting a usable brand column from `CarName`.

**Input:** `../data/car-price-assignment.csv`  
**Output:** `../data/car-price-cleaned.csv`

### 1 - Imports

In [1]:
# Core Libraries
import pandas as pd

### 2 - Load Data

In [2]:
df = pd.read_csv("../data/car-price-assignment.csv")
print(f"Starting shape: {df.shape}")
df.head()

Starting shape: (205, 26)


,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0


### 3 - Drop Unnecessary Columns

`car_ID` is just a row index with no predictive value.

In [3]:
df = df.drop(columns=["car_ID"])
print(f"Shape after dropping car_ID: {df.shape}")

Shape after dropping car_ID: (205, 25)


### 4 - Extract Brand from CarName

`CarName` contains both brand and model (e.g. "alfa-romero giulia"). With 147 unique values across 205 rows, it's too high-cardinality to use directly. We extract just the brand, which is the first word.

There are also several misspellings in the brand names that need correcting.

In [4]:
# Extract brand as the first word, lowercased
df["brand"] = df["CarName"].str.lower().str.split().str[0]

# Check unique brands before correction
print(f"Unique brands before correction: {df['brand'].nunique()}")
print(sorted(df["brand"].unique()))

Unique brands before correction: 27
['alfa-romero', 'audi', 'bmw', 'buick', 'chevrolet', 'dodge', 'honda', 'isuzu', 'jaguar', 'maxda', 'mazda', 'mercury', 'mitsubishi', 'nissan', 'peugeot', 'plymouth', 'porcshce', 'porsche', 'renault', 'saab', 'subaru', 'toyota', 'toyouta', 'vokswagen', 'volkswagen', 'volvo', 'vw']


In [5]:
# Fix misspellings
brand_corrections = {
    "alfa-romero": "alfa-romeo",
    "maxda": "mazda",
    "porcshce": "porsche",
    "toyouta": "toyota",
    "vokswagen": "volkswagen",
    "vw": "volkswagen",
}

df["brand"] = df["brand"].replace(brand_corrections)

print(f"Unique brands after correction: {df['brand'].nunique()}")
print(sorted(df["brand"].unique()))

Unique brands after correction: 22
['alfa-romeo', 'audi', 'bmw', 'buick', 'chevrolet', 'dodge', 'honda', 'isuzu', 'jaguar', 'mazda', 'mercury', 'mitsubishi', 'nissan', 'peugeot', 'plymouth', 'porsche', 'renault', 'saab', 'subaru', 'toyota', 'volkswagen', 'volvo']


In [6]:
# Drop original CarName now that we have a clean brand column
df = df.drop(columns=["CarName"])

### 5 - Convert Text Numbers to Numeric

`doornumber` and `cylindernumber` are stored as words ("two", "four") instead of integers. Converting them makes them usable as numeric features.

In [7]:
word_to_num = {
    "two": 2, "three": 3, "four": 4, "five": 5,
    "six": 6, "eight": 8, "twelve": 12,
}

df["doornumber"] = df["doornumber"].map(word_to_num)
df["cylindernumber"] = df["cylindernumber"].map(word_to_num)

# Verify no unmapped values slipped through
print(f"doornumber nulls after mapping: {df['doornumber'].isnull().sum()}")
print(f"cylindernumber nulls after mapping: {df['cylindernumber'].isnull().sum()}")
print(f"\ndoornumber values: {sorted(df['doornumber'].unique())}")
print(f"cylindernumber values: {sorted(df['cylindernumber'].unique())}")

doornumber nulls after mapping: 0
cylindernumber nulls after mapping: 0

doornumber values: [np.int64(2), np.int64(4)]
cylindernumber values: [np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(8), np.int64(12)]


### 6 - Handle Near-Zero Variance

`enginelocation` has 202 out of 205 values as "front". A feature where 98.5% of values are identical provides almost no signal for the model. We drop it.

In [8]:
print(f"enginelocation value counts:\n{df['enginelocation'].value_counts()}")
df = df.drop(columns=["enginelocation"])

enginelocation value counts:
enginelocation
front    202
rear       3
Name: count, dtype: int64


### 7 - Check for Duplicates

In [9]:
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")

Duplicate rows: 0


### 8 - Verify Final State

In [10]:
print(f"Final shape: {df.shape}")
print(f"\nColumn types:")
print(df.dtypes)
print(f"\nNull values: {df.isnull().sum().sum()}")

Final shape: (205, 24)

Column types:
symboling             int64
fueltype                str
aspiration              str
doornumber            int64
carbody                 str
drivewheel              str
wheelbase           float64
carlength           float64
carwidth            float64
carheight           float64
curbweight            int64
enginetype              str
cylindernumber        int64
enginesize            int64
fuelsystem              str
boreratio           float64
stroke              float64
compressionratio    float64
horsepower            int64
peakrpm               int64
citympg               int64
highwaympg            int64
price               float64
brand                object
dtype: object

Null values: 0


In [11]:
df.head()

,symboling,fueltype,aspiration,doornumber,carbody,drivewheel,wheelbase,carlength,carwidth,carheight,...,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price,brand
0,3,gas,std,2,convertible,rwd,88.6,168.8,64.1,48.8,...,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0,alfa-romeo
1,3,gas,std,2,convertible,rwd,88.6,168.8,64.1,48.8,...,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0,alfa-romeo
2,1,gas,std,2,hatchback,rwd,94.5,171.2,65.5,52.4,...,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0,alfa-romeo
3,2,gas,std,4,sedan,fwd,99.8,176.6,66.2,54.3,...,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0,audi
4,2,gas,std,4,sedan,4wd,99.4,176.6,66.4,54.3,...,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0,audi


### 9 - Save Cleaned Data

In [12]:
df.to_csv("../data/car-price-cleaned.csv", index=False)
print("Saved to ../data/car-price-cleaned.csv")

Saved to ../data/car-price-cleaned.csv


### 10 - Summary

**Changes made (26 → 24 columns, 205 rows unchanged):**
- Dropped `car_ID` (row index, no predictive value) and `enginelocation` (98.5% "front", near-zero variance).
- Extracted `brand` from `CarName`, correcting 6 misspellings (27 → 22 unique brands). Dropped original `CarName`.
- Converted `doornumber` and `cylindernumber` from text words to integers.
- No missing values and no duplicate rows.